# E-commerce RFM & Sales Analysis (Portfolio-Ready)

This notebook demonstrates advanced analytics for a Junior Data Analyst role in North India’s 2026 tech ecosystem. It includes RFM segmentation, cohort analysis, and interactive sales dashboards, with a focus on business KPIs and real-world impact.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import os

## 1. Load Data
We use the cleaned merged orders and customer data.

In [ ]:
# Set paths
DATA_PATH = '../data'
orders = pd.read_csv(os.path.join(DATA_PATH, 'merged_orders_cleaned.csv'))
customers = pd.read_csv(os.path.join(DATA_PATH, 'CUSTOMERS.csv'))
orders['Order Date'] = pd.to_datetime(orders['Order Date'], errors='coerce')
orders.head()

## 2. RFM Analysis & Segmentation
We calculate Recency, Frequency, and Monetary value for each customer, then segment them into 5 groups for targeted marketing.

In [ ]:
# RFM Calculation
snapshot_date = orders['Order Date'].max() + pd.Timedelta(days=1)
rfm = orders.groupby('CustomerID').agg({
    'Order Date': lambda x: (snapshot_date - x.max()).days,
    'Order ID': 'count',
    'Amount': 'sum'
}).rename(columns={'Order Date': 'Recency', 'Order ID': 'Frequency', 'Amount': 'Monetary'})
# RFM Segmentation
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])
rfm['RFM_Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm['RFM_Score'] = rfm[['R_Score','F_Score','M_Score']].astype(int).sum(axis=1)
# Assign segments
rfm['Segment'] = pd.qcut(rfm['RFM_Score'], 5, labels=['Lost','At Risk','Need Attention','Loyal','Champions'])
rfm.head()

### Segment Distribution
Visualize the number of customers in each segment.

In [ ]:
fig = px.bar(rfm['Segment'].value_counts().sort_index(),
             title='Customer Segments by RFM',
             labels={'value':'Number of Customers','index':'Segment'})
fig.show()

## 3. Cohort Analysis
Analyze customer retention by cohort (month of first purchase).

In [ ]:
orders['CohortMonth'] = orders.groupby('CustomerID')['Order Date'].transform('min').dt.to_period('M')
orders['OrderMonth'] = orders['Order Date'].dt.to_period('M')
cohort_data = orders.groupby(['CohortMonth', 'OrderMonth']).agg({'CustomerID':'nunique'}).reset_index()
cohort_pivot = cohort_data.pivot(index='CohortMonth', columns='OrderMonth', values='CustomerID').fillna(0)
cohort_pivot

## 4. Interactive Sales Dashboard
Visualize sales trends, top products, and regional performance.

In [ ]:
# Monthly Sales Trend
monthly_sales = orders.groupby(orders['Order Date'].dt.to_period('M'))['Amount'].sum().reset_index()
monthly_sales['Order Date'] = monthly_sales['Order Date'].astype(str)
fig = px.line(monthly_sales, x='Order Date', y='Amount', title='Monthly Sales Trend')
fig.show()

In [ ]:
# Top 10 Products
if 'Product Name' in orders.columns:
    top_products = orders.groupby('Product Name')['Amount'].sum().nlargest(10).reset_index()
    fig = px.bar(top_products, x='Amount', y='Product Name', orientation='h', title='Top 10 Products by Sales')
    fig.show()

In [ ]:
# Regional Sales (North India focus)
if 'State' in orders.columns:
    north_states = ['Delhi', 'Haryana', 'Punjab', 'Uttar Pradesh', 'Uttarakhand', 'Himachal Pradesh', 'Jammu & Kashmir', 'Chandigarh']
    north_sales = orders[orders['State'].isin(north_states)].groupby('State')['Amount'].sum().reset_index()
    fig = px.bar(north_sales, x='State', y='Amount', title='Sales by State (North India)')
    fig.show()

## 5. Business Impact & Insights
- RFM segmentation enables targeted marketing, improving retention and ROI.
- Cohort analysis helps track customer loyalty and churn.
- Dashboards provide actionable insights for business growth.

**This notebook is portfolio-ready and demonstrates the skills required for Junior Data Analyst roles in North India’s high-growth sectors.**